# Retrieval Augmented Generation
## Step 4.1: Semantic search and Reranker

- Semantic search find the chunks that have closest semantic meaning with the input query
- After semantic search, apply reranker to get the best result

<img src="img/RAG.JPG" width="2000"/>

In [ ]:
from langchain_ollama.embeddings import OllamaEmbeddings
embedding_model = OllamaEmbeddings(model="mxbai-embed-large:335m") 

In [2]:
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.

Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.

Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades"""

# Split into a list of sentences
texts = text.split('.')

# Clean up to remove empty spaces and new lines
texts = [t.strip(' \n') for t in texts]

In [3]:
len(texts) # total number of chunks

15

### Semantic similarity search using FAISS
- ```similarity_search_with_score``` search the top k chunks that has similar semantic to query, it also gives the corresponding score
- ```as_retriever``` do the same thing

In [4]:
from langchain_community.vectorstores import FAISS

db_faiss = FAISS.from_texts(
    texts,embedding_model
)

#### **Semantic similarity search using FAISS: similarity_search**
Use **similarity_search** when:
- Your chunks are already clean (good splitter + overlap)
- You want maximum relevance
- Your docs are short or not repetitive
- You’re going to rerank (cross-encoder) anyway
  
Use **similarity_search_with_score** when:
- You want to calculate associate score for each chunk

In [5]:
query = "when was the movie released and what is its gross?"
docs_and_scores = db_faiss.similarity_search_with_score(query, k=3)

In [6]:
# Interpreting the score: Lower the better
for doc, score in docs_and_scores:
    print(score, doc.page_content)

0.53161037 In the United States, it was first released on film stock, expanding to venues using digital projectors
0.68172836 The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014
0.70680773 Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm


- The **score** is actually L2 distance (Euclidean distance) that has the range from 0 → +∞
- Lower score is better, and always ranked top

#### **Semantic similarity search using FAISS: as_retriever**


In [8]:
retriever = db_faiss.as_retriever(search_kwargs={"k": 3})
docs_found = retriever.invoke(query)

In [11]:
for i,doc in enumerate(docs_found):
    print(i)
    print(doc.page_content)

0
In the United States, it was first released on film stock, expanding to venues using digital projectors
1
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014
2
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm


#### **Semantic similarity search using FAISS: max_marginal_relevance: MMR**
Using MMR when:
- Results contain near-duplicate chunks
- The answer spans multiple sections/pages
- You want broader context coverage (policies, manuals, textbooks)
- PDFs have repeated headers/footers / boilerplate

In [12]:
mmr_docs = db_faiss.max_marginal_relevance_search(
    query=query,
    k=3,          # final results returned
    fetch_k=20,   # candidates pulled from FAISS first
    lambda_mult=0.5  # relevance vs diversity
)


In [14]:
for i,doc in enumerate(mmr_docs):
    print(i)
    print(doc.page_content)

0
In the United States, it was first released on film stock, expanding to venues using digital projectors
1
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014
2
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm


### Reranker
- After semantic search, it's always recommended to again rerank the chunks order with the query
- For better answer quality, precision, and trustworthiness, it’s one of the highest-return on investment upgrades you can add to a RAG system.
- We need to use CrossEncoder from HuggingFace


In [15]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
pairs = [(query, d[0].page_content) for d in docs_and_scores]
scores = reranker.predict(pairs)

reranked_docs = [
    doc for _, doc in sorted(
        zip(scores, docs_and_scores),
        key=lambda x: x[0],
        reverse=True
    )
]

top_docs = reranked_docs[:5]


In [33]:
for i,doc in enumerate(top_docs):
    print("reranking", i)
    print(doc[0].page_content)
    print("oldscore", doc[1])

reranking 0
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014
oldscore 0.68172836
reranking 1
In the United States, it was first released on film stock, expanding to venues using digital projectors
oldscore 0.53161037
reranking 2
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm
oldscore 0.70680773
